# Human Delegation Provenance with HDP

When Claude operates inside a multi-agent pipeline, every action it takes is authorised by a human — but that authorisation is implicit. There is no cryptographic record of who approved the task, what scope was granted, or how the delegation flowed through the system.

**HDP (Human Delegation Provenance Protocol)** makes that authorisation explicit and tamper-evident. A human signs a root token with Ed25519; each agent that receives and delegates work extends the chain with a signed hop; any downstream consumer can verify the full chain **fully offline** — no registry, no network call.

This notebook walks through the complete lifecycle:

1. Generate a key pair (the human's signing identity)
2. Issue a root token (the human authorisation event)
3. Pass the token to a Claude agent and have it act within scope
4. Extend the chain as the task delegates further
5. Verify the full chain offline

**Package:** [`@helixar_ai/hdp`](https://www.npmjs.com/package/@helixar_ai/hdp)  
**IETF draft:** [draft-helixar-hdp-agentic-delegation-00](https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/)  
**Spec:** [helixar.ai/about/labs/hdp](https://helixar.ai/about/labs/hdp/)

## Setup

HDP is a TypeScript/Node.js library. This notebook uses the Python `anthropic` SDK for the Claude calls and shells out to Node.js for HDP operations. Both must be installed.

In [ ]:
# Install the Python Anthropic SDK if not already installed
%pip install anthropic --quiet

In [ ]:
# Install HDP (requires Node.js >= 18)
import subprocess
result = subprocess.run(['npm', 'install', '-g', '@helixar_ai/hdp'], capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
import os
import json
import subprocess
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

## Helper: run HDP operations via Node.js

We use a small inline Node.js helper so the notebook stays self-contained.

In [ ]:
def run_node(script: str) -> dict:
    """Run an inline Node.js script and return parsed JSON output."""
    result = subprocess.run(
        ['node', '--input-type=module'],
        input=script,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Node error: {result.stderr}")
    return json.loads(result.stdout)

## Step 1 — Generate a key pair

In production this key lives in a secrets manager or HSM. For this demo we generate it in memory.

In [ ]:
keys = run_node("""
import { generateKeyPair, exportPrivateKey, exportPublicKey } from '@helixar_ai/hdp';

const { privateKey, publicKey } = await generateKeyPair();
console.log(JSON.stringify({
  privateKey: exportPrivateKey(privateKey),
  publicKey:  exportPublicKey(publicKey),
}));
""")

print(f"Public key (base64url): {keys['publicKey'][:32]}...")
print("Key pair generated ✓")

## Step 2 — Issue a root token

The root token is the human authorisation event. It records:
- **Who** is authorising (`principal`)
- **What** they approved (`scope.intent`, `authorized_tools`)
- **Constraints** (`max_hops`, `network_egress`, `data_classification`)

The token is signed with Ed25519 over the canonical JSON of the header + principal + scope.

In [ ]:
SESSION_ID = 'sess-cookbook-demo-001'

token_json = run_node(f"""
import {{ issueToken, importPrivateKey }} from '@helixar_ai/hdp';

const privateKey = importPrivateKey('{keys['privateKey']}');

const token = await issueToken({{
  sessionId: '{SESSION_ID}',
  principal: {{
    id: 'usr_alice_opaque',
    id_type: 'opaque',
    display_name: 'Alice Chen',
  }},
  scope: {{
    intent: 'Research recent AI safety papers and summarise key findings.',
    authorized_tools: ['web_search', 'file_write'],
    data_classification: 'internal',
    network_egress: true,
    persistence: true,
    max_hops: 3,
  }},
  signingKey: privateKey,
  keyId: 'alice-key-v1',
  expiresInMs: 3600000,  // 1 hour
}});

console.log(JSON.stringify(token));
""")

print(f"Token ID:      {token_json['header']['token_id']}")
print(f"Session ID:    {token_json['header']['session_id']}")
print(f"Principal:     {token_json['principal']['display_name']}")
print(f"Intent:        {token_json['scope']['intent']}")
print(f"Max hops:      {token_json['scope']['max_hops']}")
print(f"Chain length:  {len(token_json['chain'])}")
print("Root token issued ✓")

## Step 3 — Claude acts as the orchestrator agent

We pass the HDP token to Claude in the system prompt. Claude can inspect the scope, confirm it is operating within the authorised intent, and produce output accordingly.

In a production system the token would travel in an `X-HDP-Token` HTTP header or an MCP context field.

In [ ]:
# Build a system prompt that includes the HDP delegation context
hdp_context = json.dumps({
    'token_id':    token_json['header']['token_id'],
    'session_id':  token_json['header']['session_id'],
    'principal':   token_json['principal']['display_name'],
    'intent':      token_json['scope']['intent'],
    'tools':       token_json['scope']['authorized_tools'],
    'hops_used':   len(token_json['chain']),
    'hops_max':    token_json['scope']['max_hops'],
}, indent=2)

system_prompt = f"""You are an AI research orchestrator operating under a human delegation token.

Your delegation context (HDP token summary):
{hdp_context}

You MUST:
- Only perform actions within the stated intent and authorized_tools
- Not exceed max_hops remaining delegation steps
- Acknowledge the delegating human in your response

If asked to do something outside your authorized scope, decline and explain why."""

response = client.messages.create(
    model='claude-opus-4-5',
    max_tokens=512,
    system=system_prompt,
    messages=[{
        'role': 'user',
        'content': 'What is your delegation scope and what will you do next?'
    }]
)

orchestrator_response = response.content[0].text
print("Claude (orchestrator):")
print(orchestrator_response)

## Step 4 — Extend the chain

As the orchestrator delegates to a sub-agent (e.g. a search agent), it adds a signed hop to the chain. This creates an auditable record that the orchestrator received the root delegation and passed it on.

In [ ]:
import json as _json

token_str = _json.dumps(token_json).replace("'", "\\'")  # escape for JS template

token_hop1 = run_node(f"""
import {{ extendChain, importPrivateKey }} from '@helixar_ai/hdp';

const privateKey = importPrivateKey('{keys['privateKey']}');
let token = {_json.dumps(token_json)};

token = await extendChain(token, {{
  agent_id:       'claude-orchestrator-v1',
  agent_type:     'orchestrator',
  action_summary: 'Decompose research task and delegate to search sub-agent.',
  parent_hop:     0,
}}, privateKey);

console.log(JSON.stringify(token));
""")

print(f"Chain length after hop 1: {len(token_hop1['chain'])}")
print(f"Hop agent:   {token_hop1['chain'][0]['agent_id']}")
print(f"Hop action:  {token_hop1['chain'][0]['action_summary']}")
print("Orchestrator hop added ✓")

In [ ]:
# Sub-agent hop — search agent receives the delegation
token_hop2 = run_node(f"""
import {{ extendChain, importPrivateKey }} from '@helixar_ai/hdp';

const privateKey = importPrivateKey('{keys['privateKey']}');
let token = {_json.dumps(token_hop1)};

token = await extendChain(token, {{
  agent_id:       'search-agent-v1',
  agent_type:     'sub-agent',
  action_summary: 'Execute web_search for recent AI safety papers.',
  parent_hop:     1,
}}, privateKey);

console.log(JSON.stringify(token));
""")

print(f"Chain length after hop 2: {len(token_hop2['chain'])}")
print(f"Hop agent:   {token_hop2['chain'][1]['agent_id']}")
print(f"Hop action:  {token_hop2['chain'][1]['action_summary']}")
print("Search agent hop added ✓")

## Step 5 — Verify the full chain (offline)

Any recipient of the token — at any point in the chain — can verify it with only:
- The issuer's Ed25519 public key (32 bytes)
- The current session ID
- The current time (for expiry)

No network call. No registry lookup.

In [ ]:
verification = run_node(f"""
import {{ verifyToken, importPublicKey }} from '@helixar_ai/hdp';

const publicKey = importPublicKey('{keys['publicKey']}');
const token = {_json.dumps(token_hop2)};

const result = await verifyToken(token, {{
  publicKey,
  currentSessionId: '{SESSION_ID}',
}});

console.log(JSON.stringify({{
  valid:       result.valid,
  chain_hops:  token.chain.length,
  principal:   token.principal.display_name,
  intent:      token.scope.intent,
  error:       result.error ?? null,
}}));
""")

print(f"Valid:       {verification['valid']}")
print(f"Chain hops:  {verification['chain_hops']}")
print(f"Principal:   {verification['principal']}")
print(f"Intent:      {verification['intent']}")
if verification['error']:
    print(f"Error:       {verification['error']}")

assert verification['valid'], "Chain verification failed!"
print("\nChain verified offline ✓ — no network calls made")

## Summary

| Step | What happened |
|------|---------------|
| 1 | Generated an Ed25519 key pair (the human's signing identity) |
| 2 | Issued a root token binding Alice's identity to a specific intent and scope |
| 3 | Claude received the token, confirmed its delegation context, and acted within scope |
| 4 | Two agent hops were added to the chain — orchestrator → search agent |
| 5 | The full chain was verified offline with only the public key and session ID |

### What this gives you

- **Auditability** — every action is traceable to the authorising human
- **Tamper-evidence** — any modification to the token or chain breaks the Ed25519 signatures
- **Offline verification** — works in air-gapped environments, edge runtimes, or anywhere a network call before every agent action is unacceptable
- **Scope enforcement hooks** — the `authorized_tools` and `data_classification` fields give application layers a structured basis for runtime policy checks

### Next steps

- **MCP integration:** use `@helixar_ai/hdp-mcp` to attach HDP tokens to MCP tool calls automatically
- **Re-authorisation:** use `issueReAuthToken()` when `max_hops` is exhausted or scope needs to expand
- **Multi-principal:** use `verifyPrincipalChain()` for joint human authorisation (e.g. two-person approval)
- **Key management:** use `KeyRegistry` with `/.well-known/hdp-keys.json` for automated key distribution

**Full spec:** [helixar.ai/about/labs/hdp](https://helixar.ai/about/labs/hdp/)  
**IETF draft:** [draft-helixar-hdp-agentic-delegation-00](https://datatracker.ietf.org/doc/draft-helixar-hdp-agentic-delegation/)